# Extract Birdcall Chirp

Load `data/raw/sample.mp3`, remove low-frequency content, and extract a 250 ms chirp around the highest-energy frame.

In [1]:
from __future__ import annotations

from pathlib import Path

import librosa
import numpy as np
import soundfile as sf
from IPython.display import Audio, display
from scipy.signal import butter, filtfilt

from voice_to_birdcall.paths import PROCESSED_DATA_DIR, RAW_DATA_DIR

SR = 44_100
CHIRP_DURATION = 0.25

bird_source = RAW_DATA_DIR / "sample.mp3"
chirp_path = PROCESSED_DATA_DIR / "birdcall-chirp.wav"

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

# Load and Filter Bird Audio

Normalize the bird sample by peak amplitude and apply a zero-phase fourth-order high-pass filter at 1 kHz.

In [2]:
def finite_audio(y: np.ndarray) -> np.ndarray:
    y = np.asarray(y, dtype=np.float32)
    if not np.all(np.isfinite(y)):
        y = np.nan_to_num(y, nan=0.0, posinf=0.0, neginf=0.0)
    return y


def normalize_audio(y: np.ndarray) -> np.ndarray:
    y = finite_audio(y)
    peak = float(np.max(np.abs(y))) if len(y) else 0.0
    return y if peak < 1e-8 else y / peak


def highpass_filter(y: np.ndarray, cutoff: float) -> np.ndarray:
    if len(y) < 32:
        return y
    b, a = butter(4, cutoff / (SR / 2), btype="highpass")
    padlen = min(3 * max(len(a), len(b)), len(y) - 1)
    return filtfilt(b, a, y, padlen=padlen)


if not bird_source.exists():
    raise FileNotFoundError(f"Bird source not found: {bird_source}")

bird, _ = librosa.load(str(bird_source), sr=SR, mono=True)
if len(bird) == 0:
    raise ValueError(f"Audio file is empty: {bird_source.name}")
bird = normalize_audio(highpass_filter(normalize_audio(bird), cutoff=1_000))

# Extract and Save Chirp

Center the source window near the highest-RMS frame and apply 20 ms fades at both boundaries before saving `data/processed/birdcall-chirp.wav`.

In [3]:
def make_chirp_from_bird(
    bird_audio: np.ndarray,
    duration: float = CHIRP_DURATION,
) -> np.ndarray:
    length = int(duration * SR)
    if len(bird_audio) < length:
        bird_audio = np.pad(bird_audio, (0, length - len(bird_audio)))

    rms = librosa.feature.rms(y=bird_audio, frame_length=2048, hop_length=512)[0]
    max_frame = int(np.argmax(rms))
    start = max(0, max_frame * 512 - length // 4)
    end = start + length
    if end > len(bird_audio):
        start, end = max(0, len(bird_audio) - length), len(bird_audio)

    chirp = bird_audio[start:end].copy()
    if len(chirp) < length:
        chirp = np.pad(chirp, (0, length - len(chirp)))

    fade_len = min(int(0.02 * SR), len(chirp) // 2)
    if fade_len > 0:
        chirp[:fade_len] *= np.linspace(0, 1, fade_len)
        chirp[-fade_len:] *= np.linspace(1, 0, fade_len)
    return normalize_audio(chirp)


chirp = make_chirp_from_bird(bird)
sf.write(str(chirp_path), chirp, SR, subtype="PCM_16")

print(f"Saved birdcall chirp to: {chirp_path}")
display(Audio(chirp, rate=SR))

Saved birdcall chirp to: C:\Users\morph\GitHub\voice-to-birdcall\data\processed\birdcall-chirp.wav
